# Tutorial 04 - Exercise 04: Hospital Visualization with Matplotlib vs Seaborn

Builds the nine charts of the Exercise 04 handout on the risk-scored hospital
cohort produced by notebook 02 (`data/hospital_risk.csv`): four Matplotlib
charts drawn by hand (bar + annotations, histogram + reference lines, scatter
with highlights, a 1x3 panel grid with a shared y axis) and five Seaborn
statistical charts (grouped bar, box + violin, correlation heatmap, pairplot
and a 2x2 overview dashboard). Every figure is audited in memory: a figure
that renders blank or without data would fail the audit cell.

---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/06_Hospital_Department_Analytics/` project folder (anchored on `data/hospital_original.csv`; exports also go to the project's `data/` folder) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "06_Hospital_Department_Analytics"
ANCHOR_REL = "data/hospital_original.csv"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Load and Inspect the Cohort

The cohort is the 8-patient table of Exercise 03 after its risk pass (notebook 02):
`TotalCost`, `AgeGroup`, `RiskScore`, `RiskCategory` and the flag columns are
already computed, so this notebook only reads and draws.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

df = pd.read_csv(PROJ_DIR / "data" / "hospital_risk.csv")
print(df.shape)
df.info()
df[["DaysAdmitted", "DailyFee", "TotalCost", "Satisfaction", "RiskScore"]].describe().round(2)

### 2. Figure Audit Helper

Each chart below is rendered to an in-memory PNG and measured instead of being
saved to disk (the repository commits no image files). A figure counts as drawn
when it has a non-blank pixel ratio above 1% and more than ten distinct colours,
and the audit prints the three numbers so a blank render cannot pass silently.
The helper closes every figure after auditing it, so the notebook stays light
and the figures never leak into the next cell.

In [ ]:
import io

from PIL import Image

AUDITS = []

def audit(fig, name):
    """Render fig to an in-memory PNG, measure it, record the audit, close it."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    pixels = np.asarray(image)
    non_white = (pixels < 250).any(axis=2).mean()
    colors = {tuple(c) for row in pixels[::4] for c in row[::4]}
    AUDITS.append((name, non_white, len(colors), buf.getbuffer().nbytes))
    plt.close(fig)
    print(f"{name:34s} non-blank {non_white:6.1%} | distinct colors {len(colors):4d} | {buf.getbuffer().nbytes:6d} bytes")
    return fig

### 3. Matplotlib Part

#### Chart 1 - Total Cost per City (bar, annotations, custom colors)

The bars are the city averages of `TotalCost`. Annotations write the exact
value over each bar, which is what the handout calls "annotate each bar with
its total". The y axis starts at zero so the height stays proportional to the
value, and the limit leaves headroom for the annotation above the tallest bar.

In [ ]:
palette = {"Alger": "#4C72B0", "Annaba": "#DD8452", "Oran": "#55A868"}
city_costs = df.groupby("City")["TotalCost"].mean().round(1)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(city_costs.index, city_costs.values,
              color=[palette[c] for c in city_costs.index], edgecolor="black")
for rect, value in zip(bars, city_costs.values):
    ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 40,
            f"{value:,.0f}", ha="center", va="bottom", fontweight="bold")
ax.set_title("Average Total Cost per City (TP4 Exercise 04)", fontweight="bold")
ax.set_xlabel("City")
ax.set_ylabel("Average TotalCost")
ax.set_ylim(0, city_costs.max() * 1.18)
audit(fig, "M1 bar: cost per city")

#### Chart 2 - Satisfaction Histogram with Mean and Median Lines

A histogram of the eight satisfaction scores with the mean and the median drawn
as vertical reference lines. The bin count is the histogram's most visible
parameter, so the sensitivity row draws the same data with 3, 4 and 5 bins:
with eight observations the shape changes completely from one bin count to the
next, and only the reference lines stay meaningful. The 8-row cohort is a
reminder that a histogram is a rough density estimate on small samples.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["Satisfaction"], bins=4, color="#4C72B0", edgecolor="white")
ax.axvline(df["Satisfaction"].mean(), color="#DD8452", linestyle="--", linewidth=2,
           label=f"Mean {df['Satisfaction'].mean():.2f}")
ax.axvline(df["Satisfaction"].median(), color="#55A868", linestyle=":", linewidth=2,
           label=f"Median {df['Satisfaction'].median():.2f}")
ax.set_title("Satisfaction Distribution with Mean / Median", fontweight="bold")
ax.set_xlabel("Satisfaction")
ax.set_ylabel("Stays")
ax.legend()
audit(fig, "M2 histogram + vlines")

fig, axs = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, bins in zip(axs, (3, 4, 5)):
    ax.hist(df["Satisfaction"], bins=bins, color="#4C72B0", edgecolor="white")
    ax.set_title(f"bins={bins}")
    ax.set_xlabel("Satisfaction")
axs[0].set_ylabel("Stays")
fig.suptitle("Bin sensitivity (8 observations)", fontweight="bold")
audit(fig, "M2b histogram bin sensitivity")

#### Chart 3 - TotalCost vs Satisfaction Scatter, High-Cost Stays Highlighted

Each stay is one point. The two stays above the 75th percentile of cost are drawn
larger and in a warning colour, and the annotation names the single most expensive
stay. The scatter makes the absence of a strong relation visible: the costliest
stay (Oran, Neurology, 2610) is not among the least satisfying ones.

In [ ]:
threshold = df["TotalCost"].quantile(0.75)
high = df["TotalCost"] > threshold

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df.loc[~high, "TotalCost"], df.loc[~high, "Satisfaction"],
           s=70, color="#4C72B0", edgecolor="black", label="stay")
ax.scatter(df.loc[high, "TotalCost"], df.loc[high, "Satisfaction"],
           s=150, color="#C44E52", edgecolor="black", marker="D",
           label=f"high cost (> p75 = {threshold:.0f})")
costliest = df.loc[df["TotalCost"].idxmax()]
ax.annotate(f"{costliest['City']} / {costliest['Department']}\n{int(costliest['TotalCost']):,d}",
            xy=(costliest["TotalCost"], costliest["Satisfaction"]),
            xytext=(costliest["TotalCost"] * 0.86, costliest["Satisfaction"] + 0.28),
            arrowprops=dict(arrowstyle="->", color="black"))
ax.set_title("Total Cost vs Satisfaction (high-cost stays highlighted)", fontweight="bold")
ax.set_xlabel("TotalCost")
ax.set_ylabel("Satisfaction")
ax.legend(loc="lower left")
audit(fig, "M3 scatter + highlight")

#### Chart 4 - City x Department Satisfaction (1x3 panels, shared y axis)

One panel per city, average satisfaction per department, all panels on one y
axis (`sharey=True`) so the bars are comparable across cities. This is the
grid the handout asks for as "one row, three columns, same scale".

In [ ]:
cities = ["Alger", "Annaba", "Oran"]

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, city in zip(axs, cities):
    per_dept = df[df["City"] == city].groupby("Department")["Satisfaction"].mean().round(2)
    ax.bar(per_dept.index, per_dept.values, color=palette[city], edgecolor="black")
    ax.set_title(city, fontweight="bold")
    ax.set_xlabel("Department")
    for rect, value in zip(ax.patches, per_dept.values):
        ax.text(rect.get_x() + rect.get_width() / 2, value + 0.04,
                f"{value:.2f}", ha="center", va="bottom", fontsize=9)
axs[0].set_ylabel("Mean satisfaction")
fig.suptitle("Mean Satisfaction per Department and City (shared y axis)", fontweight="bold")
audit(fig, "M4 1x3 panels sharey")

### 4. Seaborn Part

#### Chart 5 - Mean RiskScore per City, split by Department (`sns.barplot`)

`barplot` computes the aggregation itself: one rectangle per (city, department)
group, the mean of `RiskScore`, and the thin error line is the 95% bootstrap
confidence interval of that mean. A `plt.bar` call would need the `groupby`
step first - the aggregation is the difference between the two libraries.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=df, x="City", y="RiskScore", hue="Department", ax=ax,
            palette="Set2", errorbar=("ci", 95))
ax.set_title("Mean RiskScore per City, split by Department", fontweight="bold")
ax.set_xlabel("City")
ax.set_ylabel("Mean RiskScore (95% CI)")
ax.legend(title="Department", loc="upper right")
audit(fig, "S5 barplot hue=Department")

#### Chart 6 - RiskScore Distribution: Box and Violin Side by Side

Same variable, two views. The box plot gives the five-number summary (median
line, box = interquartile range, whiskers = 1.5 IQR, points = outliers); the
violin draws a kernel density on each side, which is smoother but invents
shape on an 8-row sample. The handout asks for both precisely to compare the
two readings of the same column.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=df, x="City", y="RiskScore", palette=palette, ax=axs[0])
axs[0].set_title("RiskScore per City - box plot", fontweight="bold")
sns.violinplot(data=df, x="City", y="RiskScore", palette=palette, inner="box", ax=axs[1])
axs[1].set_title("RiskScore per City - violin plot", fontweight="bold")
for ax in axs:
    ax.set_xlabel("City")
    ax.set_ylabel("RiskScore")
audit(fig, "S6 box + violin RiskScore")

#### Chart 7 - Correlation Heatmap

The numeric columns only: the flags and labels would make `.corr()` fail or
return a meaningless 1/0 block. Satisfaction against RiskScore is the
instructive pair. Comment on the reading: the strong negative correlation
means costlier, longer and more readmitted stays score lower satisfaction -
an association, not a causal claim, on eight stays.

In [ ]:
numeric = df[["Age", "DaysAdmitted", "DailyFee", "TotalCost",
               "Satisfaction", "RiskScore", "Readmitted"]]

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(numeric.corr().round(2), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Heatmap (numeric columns)", fontweight="bold")
audit(fig, "S7 correlation heatmap")
print("Satisfaction vs RiskScore:", numeric.corr().loc["Satisfaction", "RiskScore"])

#### Chart 8 - Pairwise Relationships (`sns.pairplot`, hue=City)

`pairplot` draws every numeric column against every other one, with the
marginal distribution on the diagonal, one colour per city. It is the
one chart that needs no axes and accepts no `ax=`: it builds its own grid
(`PairGrid`), so it is saved through its own figure. The audit renders it
like any other figure.

In [ ]:
pair_cols = ["Age", "DaysAdmitted", "TotalCost", "Satisfaction", "RiskScore"]
grid = sns.pairplot(df, vars=pair_cols, hue="City", palette=palette,
                    diag_kind="hist", height=1.9, corner=False)
grid.figure.suptitle("Pairwise Relationships by City", y=1.02, fontweight="bold")
audit(grid.figure, "S8 pairplot hue=City")

#### Chart 9 - Hospital Risk and Satisfaction Overview (2x2 dashboard)

The closing dashboard: four panels, one per question - risk per city, cost
per city, satisfaction distribution, and the cost/satisfaction relation with
the readmitted stays marked. One `fig` with four `Axes` objects is the
Matplotlib canvas model (one Figure, many Axes) that the whole chapter builds
on, and this panel is where Seaborn calls sit on axes created by
`plt.subplots` through their `ax=` keyword.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 9))

sns.barplot(data=df, x="City", y="RiskScore", hue="Readmitted", ax=axs[0, 0],
            palette="Set2")
axs[0, 0].set_title("Mean RiskScore by City and Readmission", fontweight="bold")
axs[0, 0].set_xlabel("City")

city_costs = df.groupby("City")["TotalCost"].mean().round(1)
axs[0, 1].bar(city_costs.index, city_costs.values, color=[palette[c] for c in city_costs.index],
              edgecolor="black")
axs[0, 1].set_title("Average Total Cost per City", fontweight="bold")
axs[0, 1].set_xlabel("City")

axs[1, 0].hist(df["Satisfaction"], bins=4, color="#4C72B0", edgecolor="white")
axs[1, 0].axvline(df["Satisfaction"].mean(), color="#DD8452", linestyle="--", linewidth=2,
                   label=f"Mean {df['Satisfaction'].mean():.2f}")
axs[1, 0].set_title("Satisfaction Distribution", fontweight="bold")
axs[1, 0].set_xlabel("Satisfaction")
axs[1, 0].legend()

sns.scatterplot(data=df, x="TotalCost", y="Satisfaction", hue="Readmitted",
                style="City", s=90, ax=axs[1, 1])
axs[1, 1].set_title("Cost vs Satisfaction (readmitted marked)", fontweight="bold")
axs[1, 1].set_xlabel("TotalCost")

fig.suptitle("Hospital Risk and Satisfaction Overview (TP4 Exercise 04)", fontweight="bold")
fig.tight_layout()
audit(fig, "S9 2x2 overview dashboard")

### 5. Audit Summary

All nine charts of the handout are accounted for. The table is printed from the
audit records, and the assertion cell turns the audit into a gate: a chart that
renders blank, or a figure without data, fails the notebook instead of passing
silently.

In [ ]:
audit_table = pd.DataFrame(AUDITS, columns=["chart", "non_blank_ratio", "distinct_colors", "png_bytes"])
audit_table["non_blank_ratio"] = audit_table["non_blank_ratio"].round(3)
display(audit_table)

failed = audit_table[(audit_table["non_blank_ratio"] < 0.01) | (audit_table["distinct_colors"] < 10)]
assert failed.empty, f"blank or empty figure(s): {failed['chart'].tolist()}"
print(f"all {len(AUDITS)} figures drawn, audited non-blank and closed")